# 01 · Read temperature and status

Read temperature, setpoint and diagnostics. Readings are sequential, not simultaneous; compare with the front panel and an independent reference.

Run top to bottom in a **Python 3.12+** kernel. This notebook includes all configuration and setup; no sibling example files are imported. Run only one hardware notebook/process at a time.

**Prepare the filled cooling loop before connecting.** T255P connection sends U, takes serial control and can change AUTO START to RUN, even with target writes disabled. Software disconnect does not stop the pump/chiller or restore its target. Physical hardware has not been validated with this package.

## 1. Environment and imports

From the repository root, run `uv sync --locked --group notebooks`, then `uv run --locked --group notebooks jupyter lab`. Select the environment kernel described in examples/README.md. The script-only dashboard (07_dashboard.py) needs `--extra gui`. Restart the kernel after installation or driver source changes. Imports open no serial ports.

In [1]:
import sys
from pathlib import Path

# Prefer checkout sources when launched from the repository or examples folder.
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src/tark_chiller/controller.py").is_file():
        sys.path.insert(0, str(candidate / "src"))
        break

In [2]:
from tark_chiller.controller import TarkChiller
from tark_chiller.protocols import T255P, ProtocolError

## 2. Choose the port

This example connects to a **ThermoTek T255P**. Set `PORT` to its serial port (for example `"COM9"`). Adapter metadata identifies the USB adapter, not the attached chiller.

The connection cell below spells out the T255P profile: 9600 baud, 8N1, XON/XOFF, no hardware flow control, and RTS/DTR deasserted. Check that your adapter/cable supports these line levels. Distilled-water target bounds are 5–45 °C, in 0.1 °C steps. See the [hardware guide](../docs/hardware.md) for wiring and the separate CAL protocol.

In [3]:
PORT = "COM9"  # Set the identified serial port, e.g. "COM9".

## 3. Connection settings

The next operation cell constructs the driver directly with the settings shown. Construction opens nothing. Its context connects and releases the port, including on a read error or interrupt. Among the notebooks, only example 02 enables target writes; dashboard script 07 also enables explicit GUI target requests.

## 4. Read temperature and status

This lesson calls `get_status()` (U), `get_alarms()` (J), `get_temperature()` (I) and `get_memory()` (H0) explicitly. It checks the sensor alarm before I and displays named modes, boolean flags, setpoint and the raw maximum-power setting.

Existing cell outputs are retained from earlier runs; rerun the cells to see the updated output.


In [4]:
if not isinstance(PORT, str) or not PORT.strip():
    raise ValueError('Hardware not configured: set PORT, e.g. "COM9". No port was opened.')

chiller = TarkChiller(
    {
        "port": PORT,
        "baudrate": 9600,
        "bytesize": 8,
        "parity": "N",
        "stopbits": 1,
        "xonxoff": True,
        "rtscts": False,
        "dsrdtr": False,
        "rts": False,
        "dtr": False,
    },
    protocol=T255P(allow_writes=False),
    setpoint_range=(5.0, 45.0),
    coolant_source="ThermoTek T255P Rev C pp9/20; verify installed distilled water",
)

# T255P connect sends U: it takes serial control and can start AUTO START.
with chiller:
    # Each get_ call sends exactly one command. U refreshes the session;
    # J reads alarms, I reads temperature, and H0 reads target/max-power memory.
    status = chiller.get_status()  # U
    alarms = chiller.get_alarms()  # J
    print(f"Connected: {status['connected']}")
    print(f"Backend:   {status['backend']}")
    print(f"Mode:      {status['mode']} (code {status['mode_code']})")
    print(f"Chiller on: {status['chiller_on']}; dryer on: {status['dryer_on']}")
    print(f"Alarm summary: {status['summary_alarm']}")
    for name, active in alarms.items():
        print(f"  {name}: {'ACTIVE' if active else 'inactive'}")
    print(f"Details:   {status['detail']}")
    # A direct I query does not check J; make that decision explicitly here.
    if alarms["sensor"]:
        raise ProtocolError("T255P sensor alarm; temperature unavailable")
    temperature_c = chiller.get_temperature()  # I
    memory = chiller.get_memory()  # H0
    print(f"Temperature:       {temperature_c:.2f} C")
    print(f"Reported setpoint: {memory['setpoint_c']:.2f} C")
    print(f"Maximum-power setting (raw): {memory['max_power']}")
    # Compare Celsius readings with the front panel and your reference.

# 4. Context exit releases the port, including after a failed read. If cleanup
# also fails, the original error is preserved with a cleanup note. Retry cleanup
# as needed; closing the software connection does not stop equipment.

Connected: True
Backend:   hardware
Mode:      AUTO START (code 0)
Chiller on: True; dryer on: False
Alarm summary: False
  float_switch: inactive
  high_temp: inactive
  low_temp: inactive
  sensor: inactive
  eeprom: inactive
  watchdog: inactive
Details:   T255P (human-selected; identity not queried); mode=AUTO START; summary_alarm=0 chiller_on=1 dryer_on=0; flow/leak unknown; last serial reply, not safety proof
Temperature:       18.51 C
Reported setpoint: 18.00 C
Maximum-power setting (raw): 190


## 5. Check results and finish

T255P diagnostics report mode/alarm flags; they do not identify the unit.

Context cleanup closes software resources, including after a read error or kernel interrupt. If cleanup fails, resources may remain owned: allow pending I/O to finish and retry `chiller.disconnect(timeout_s=5.0)` if `chiller` was created. Do not rerun configuration over a live connection. A forced kernel termination cannot guarantee cleanup. Follow the equipment's physical shutdown procedure separately.